In [2]:
import psycopg2
from getpass import getpass

import pandas as pd
import numpy as np

In [3]:
PATH_SQL = "../../sql/schemas.sql"
PATH_PERIMETER = "../../data/food_perimeter.parquet"

In [4]:
COLS = [
    'energy-kcal_100g',
    'fat_100g',
    'saturated-fat_100g',
    'carbohydrates_100g',
    'sugars_100g',
    'fiber_100g',
    'proteins_100g',
    'salt_100g'
]

In [5]:
conn = psycopg2.connect(
    dbname="nutriscope",
    host="localhost",
    port="5432",
    user="postgres",
    password=getpass("Mot de passe : ")
)

In [6]:
# Test
cursor = conn.cursor()
# Exécuter une requête simple
cursor.execute("SELECT version();")
# Récupère une ligne de la réponse.
cursor.fetchone()

('PostgreSQL 17.11 (Debian 17.11-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)

In [7]:
# Lire le fichier sql afin de créer les tables. (remise à zéro)
with open(PATH_SQL, encoding="utf-8") as file:
    content = file.read()

# Exécuter la requête
cursor.execute(content)
conn.commit()

In [8]:
df = pd.read_parquet(PATH_PERIMETER)

In [9]:
df.shape

(456484, 14)

In [10]:
df['code'].duplicated(keep=False).sum()

np.int64(10)

In [11]:
df.iloc[df['code'].duplicated(keep=False)].notna().mean(axis=1) * 100

2484       71.428571
2489      100.000000
2490      100.000000
423876    100.000000
441449    100.000000
441538    100.000000
441539    100.000000
441541    100.000000
441591    100.000000
441592    100.000000
dtype: float64

In [12]:
# Création d'une nouvelle colonne afin de vérifier les lignes les plus remplies
df["not_na_count"] = df.notna().sum(axis=1)

In [13]:
# Retirer les doublons après les avoir organisé du plus remplie ou moins remplie, drop_duplicates() garde les premières lignes rencontrées et retire ses doublons (forcément celles remplie de la même manière ou moins bien remplie).
df = df.sort_values("not_na_count", ascending=False).drop_duplicates('code')

In [14]:
df.shape # 456484 => 456479 = 5 lignes en moins

(456479, 15)

In [15]:
def get_first_brand(brands):
    """Récupère la premère marque d'un produit"""
    if not isinstance(brands, str):
        return None
    list_brands = [" ".join(brand.split()).lower() for brand in brands.split(',')]
    if list_brands[0] == "" or list_brands == []:
        return None
    return list_brands[0]


In [16]:
# Test 
brands = "YooNuts, Carrefour"

list_brands = [" ".join(brand.split()) for brand in brands.split(',')]
list_brands[0]

'YooNuts'

In [17]:
print(get_first_brand(brands))

yoonuts


In [18]:
df['brand'] = df['brands'].apply(lambda x : get_first_brand(x))

In [19]:
df['brand'].nunique() # 41 339

41339

In [20]:
# df['nutriscore_grade'] = df['nutriscore_grade'].replace(['unknown', 'not-applicable'], np.nan)

In [21]:
# OU
df['nutriscore_grade'] = df['nutriscore_grade'].where(df['nutriscore_grade'].isin(['a','b', 'c', 'd', 'e'])) # Plus viable pour des valeurs inconnues.

In [22]:
df['nutriscore_grade'].isna().sum() # 34_497

np.int64(34497)

In [23]:
before = df[COLS].isna().mean() * 100
before

energy-kcal_100g       0.000000
fat_100g               0.456976
saturated-fat_100g     1.155365
carbohydrates_100g     0.466834
sugars_100g            0.924468
fiber_100g            47.407876
proteins_100g          0.327726
salt_100g              3.753075
dtype: float64

In [24]:
for col in COLS :
    if col == 'energy-kcal_100g' : 
        mask = (df[col] < 0) | (df[col] > 950)
    else:
        mask = (df[col] < 0) | (df[col] > 100)
    df.loc[mask, col] = np.nan

In [25]:
after = df[COLS].isna().mean() * 100
after

energy-kcal_100g       0.074702
fat_100g               0.461577
saturated-fat_100g     1.157556
carbohydrates_100g     0.474063
sugars_100g            0.929725
fiber_100g            47.420801
proteins_100g          0.330136
salt_100g              3.755485
dtype: float64

In [26]:
print("Différence de présence de np.nan")
after - before # Légère augmentation ce qui est totalement normal puisque c'est ma fonction qui a augmenté le nombre de NaN.

Différence de présence de np.nan


energy-kcal_100g      0.074702
fat_100g              0.004600
saturated-fat_100g    0.002191
carbohydrates_100g    0.007229
sugars_100g           0.005258
fiber_100g            0.012925
proteins_100g         0.002410
salt_100g             0.002410
dtype: float64

In [27]:
print(df.shape)
df = df.drop(columns=['not_na_count'])
print(df.shape)

(456479, 16)
(456479, 15)


In [28]:
brands =  df['brand'].dropna().unique()
brands = [(brand,) for brand in brands]
brands

[('yoonuts',),
 ('fiorini',),
 ('genius',),
 ('u bio',),
 ('marque repère',),
 ('whole earth',),
 ('chabiothé',),
 ('les recettes de dorothée',),
 ('les 3 chênes',),
 ('viči',),
 ('nyctea',),
 ('marks & spencer',),
 ('lindt',),
 ('beigel beigel',),
 ('macarons de pauline',),
 ('vitamalt',),
 ('lima',),
 ('aline & olivier',),
 ('dmbio',),
 ('le bio de manon',),
 ('m & s',),
 ('marks & spencers',),
 ('m&s',),
 ('nature & cie',),
 ('m&s food',),
 ('sainsburys',),
 ("reed's",),
 ("trader joe's",),
 ("mcdonald's",),
 ('marks & spencdjdj',),
 ('marks and spencer',),
 ('marks&spencer',),
 ('naturalia',),
 ('artisans noisetiers',),
 ("menguy's",),
 ('connétable',),
 ('bonduelle',),
 ('la mémère',),
 ("sainsbury's",),
 ('liebig',),
 ('marks &spencer',),
 ('conad',),
 ('grand fermage',),
 ('mannini',),
 ('thomas le prince',),
 ('nestlé',),
 ('danival',),
 ('lavazza',),
 ('carrefour',),
 ('nous anti-gaspi',),
 ('casino',),
 ("lay's",),
 ('biocoop',),
 ('disney',),
 ('del monte quality',),
 ('coun

In [29]:
len(brands)

41339

In [30]:
from psycopg2.extras import execute_values
execute_values(cursor, "INSERT INTO brands (name) VALUES %s ON CONFLICT (name) DO NOTHING", brands)
conn.commit()

In [31]:
cursor.execute("SELECT name, id FROM brands")
brands_id = dict(cursor.fetchall())
brands_id['carrefour'] # 49

49

In [32]:
food_groups = df['food_group'].unique()
food_groups = [(food_group, True) for food_group in food_groups]
food_groups

[('en:sugary-snacks', True),
 ('en:salty-snacks', True),
 ('en:cereals-and-potatoes', True),
 ('en:milk-and-dairy-products', True),
 ('en:beverages', True),
 ('en:composite-foods', True),
 ('en:fats-and-sauces', True),
 ('en:fruits-and-vegetables', True),
 ('en:fish-meat-eggs', True)]

In [33]:
categories_tags = df['categories_tags'].explode()
categories_tags = categories_tags.dropna().unique()
categories_tags = [(categorie_tag, False) for categorie_tag in categories_tags]
categories_tags

[('en:snacks', False),
 ('en:desserts', False),
 ('en:sweet-snacks', False),
 ('en:biscuits-and-cakes', False),
 ('en:pastries', False),
 ('en:baklava', False),
 ('en:plant-based-foods-and-beverages', False),
 ('en:plant-based-foods', False),
 ('en:nuts-and-their-products', False),
 ('en:nuts', False),
 ('en:mixed-nuts', False),
 ('en:pastas', False),
 ('en:gnocchi', False),
 ('en:filled-gnocchi', False),
 ('en:cereals-and-potatoes', False),
 ('en:specific-products', False),
 ('en:products-for-specific-diets', False),
 ('en:breads', False),
 ('en:products-without-gluten', False),
 ('en:gluten-free-breads', False),
 ('en:sliced-breads', False),
 ('en:gluten-free-sliced-breads', False),
 ('en:fruit-based-foods-and-beverages', False),
 ('en:fruits-and-vegetables-based-foods', False),
 ('en:fruit-based-foods', False),
 ('en:sweeteners', False),
 ('en:coconuts-and-their-products', False),
 ('en:sugars', False),
 ('en:coconut-sugar', False),
 ('en:dairies', False),
 ('en:creams', False),
 ('

In [34]:
execute_values(cursor, "INSERT INTO categories (name, main) VALUES %s ON CONFLICT (name) DO NOTHING",food_groups)
conn.commit()
# Ici l'ordre est important, pourquoi ? Car food_groups possède des rayons similaire aux catégories, pour garder le (name, True) food_groups doit être ajouté en premier autrement (name, False) prendrait le dessus.
execute_values(cursor, "INSERT INTO categories (name, main) VALUES %s ON CONFLICT (name) DO NOTHING",categories_tags)
conn.commit()

In [35]:
cursor.execute("SELECT name, id FROM categories")
categories_ids = dict(cursor.fetchall())
categories_ids['en:milk-and-dairy-products'] # 4

4

products (code, brand_id, brand, name, nutriscore_grade)

In [36]:
df['brand_id'] = df['brand'].map(brands_id)

In [37]:
df.shape

(456479, 16)

In [38]:
df['brand_id'].isna().sum() # 97_267

np.int64(97267)

In [39]:
products = df[["code", "brand_id", "brand", "product_name", "nutriscore_grade"]]
products = products.astype(object).where(products.notna(), None)

In [40]:
products.isna().sum()

code                    0
brand_id            97267
brand               97267
product_name            0
nutriscore_grade    34497
dtype: int64

In [41]:
rows = list(products.itertuples(index=False, name=None))

In [42]:
execute_values(cursor, "INSERT INTO products (code, brand_id, brand, name, nutriscore_grade) VALUES %s", rows, page_size=10_000)
conn.commit()

In [43]:
products.head()

,code,brand_id,brand,product_name,nutriscore_grade
456470,5200413402636,None,None,Baklawa with fig,e
456469,3700146213780,1.0,yoonuts,Original Mix,d
456468,3250393480096,2.0,fiorini,Gnocchi a poeler fromage jambon,c
6,0000554004509,3.0,genius,Pain de mie sans gluten,c
456483,3706228437938,4.0,u bio,Sucre de fleur de coco en poudre,e


In [44]:
categories = df[['code', 'categories_tags']]
categories = categories.explode('categories_tags')
categories = categories.astype(object).where(categories.notna(), None)
categories = categories.rename(columns={"categories_tags" : 'category'})
categories

,code,category
456470,5200413402636,en:snacks
456470,5200413402636,en:desserts
456470,5200413402636,en:sweet-snacks
456470,5200413402636,en:biscuits-and-cakes
456470,5200413402636,en:pastries
...,...,...
329562,2846163013710,en:rillettes
329562,2846163013710,en:potted-meats
329562,2846163013710,en:white-meat-rillettes
329562,2846163013710,en:poultry-rillettes


In [45]:
food_group = df[['code', 'food_group']]
food_group = food_group.astype(object).where(food_group.notna(), None)
food_group = food_group.rename(columns={"food_group" : 'category'})
food_group

,code,category
456470,5200413402636,en:sugary-snacks
456469,3700146213780,en:salty-snacks
456468,3250393480096,en:cereals-and-potatoes
6,0000554004509,en:cereals-and-potatoes
456483,3706228437938,en:sugary-snacks
...,...,...
329061,6901668009968,en:sugary-snacks
328983,2279390019949,en:fish-meat-eggs
331207,3760285580228,en:sugary-snacks
329636,1900000006585,en:sugary-snacks


In [46]:
food_group_names = df["food_group"].unique()
categories = categories[~categories["category"].isin(food_group_names)]

In [47]:
cat_concat = pd.concat([food_group, categories])
cat_concat.head()

,code,category
456470,5200413402636,en:sugary-snacks
456469,3700146213780,en:salty-snacks
456468,3250393480096,en:cereals-and-potatoes
6,0000554004509,en:cereals-and-potatoes
456483,3706228437938,en:sugary-snacks


In [48]:
cat_concat = cat_concat.drop_duplicates()

In [49]:
cat_concat['category_id'] = cat_concat['category'].map(categories_ids)

In [50]:
cat_concat = cat_concat.drop(columns=['category'])

In [51]:
cat_concat

,code,category_id
456470,5200413402636,1
456469,3700146213780,2
456468,3250393480096,3
6,0000554004509,3
456483,3706228437938,1
...,...,...
329562,2846163013710,205
329562,2846163013710,2266
329562,2846163013710,4034
329562,2846163013710,4035


In [52]:
rows = list(cat_concat[["category_id", "code"]].itertuples(index=False, name=None))
execute_values(
    cursor,
    "INSERT INTO products_categories (category_id, product_id) VALUES %s",
    rows,
    page_size=10_000,
)
conn.commit()

nutriments (id, product_id, energy, energy_u, fat, fat_u, saturated_fat, saturated_fat_u, carbohydrates, carbohydrates_u, sugars, sugars_u, fiber, fiber_u, proteins, proteins_u, salt, salt_u)

In [53]:
df.columns

Index(['code', 'product_name', 'brands', 'categories_tags', 'nutriscore_grade',
       'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g',
       'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g',
       'salt_100g', 'food_group', 'brand', 'brand_id'],
      dtype='str')

In [58]:
nutriments = df[['code','energy-kcal_100g','fat_100g','saturated-fat_100g','carbohydrates_100g','sugars_100g', 'fiber_100g','proteins_100g','salt_100g']]
nutriments = nutriments.rename(columns={
    'code':'product_id', 
    'energy-kcal_100g' : 'energy', 
    'fat_100g' : "fat",
    'saturated-fat_100g' : "saturated_fat",
    'carbohydrates_100g' : "carbohydrates",
    'sugars_100g' : 'sugars', 
    'fiber_100g' : "fiber",
    'proteins_100g' : "proteins",
    'salt_100g' : "salt"})

In [59]:
nutriments['energy_u'] = "kcal"
nutriments[['fat_u', 'saturated_fat_u', 'carbohydrates_u', 'sugars_u', 'fiber_u', 'proteins_u', 'salt_u']] = "g"

In [60]:
nutriments.head()

,product_id,energy,fat,saturated_fat,carbohydrates,sugars,fiber,proteins,salt,energy_u,fat_u,saturated_fat_u,carbohydrates_u,sugars_u,fiber_u,proteins_u,salt_u
456470,5200413402636,406.000000,9.0,2.2,72.800003,36.000000,2.5,7.2,0.950000,kcal,g,g,g,g,g,g,g
456469,3700146213780,538.000000,38.0,5.0,29.000000,23.000000,7.0,17.0,0.000000,kcal,g,g,g,g,g,g,g
456468,3250393480096,202.000000,5.2,2.7,30.000000,1.900000,2.5,7.5,1.000000,kcal,g,g,g,g,g,g,g
6,0000554004509,279.000000,6.3,0.5,48.200001,4.400000,5.8,4.4,1.090000,kcal,g,g,g,g,g,g,g
456483,3706228437938,399.398987,0.0,0.0,99.899803,99.899803,0.0,0.0,0.005405,kcal,g,g,g,g,g,g,g


In [70]:
nutriments = nutriments.astype(object).where(nutriments.notna(), None)

In [71]:
sql_cols = ['product_id','energy','energy_u', 'fat','fat_u',"saturated_fat",'saturated_fat_u','carbohydrates','carbohydrates_u','sugars','sugars_u', 'fiber', 'fiber_u', 'proteins', 'proteins_u','salt', 'salt_u']
rows = list(nutriments[sql_cols].itertuples(index=False, name=None))

In [72]:
rows

[('5200413402636',
  406.0,
  'kcal',
  9.0,
  'g',
  2.200000047683716,
  'g',
  72.80000305175781,
  'g',
  36.0,
  'g',
  2.5,
  'g',
  7.199999809265137,
  'g',
  0.949999988079071,
  'g'),
 ('3700146213780',
  538.0,
  'kcal',
  38.0,
  'g',
  5.0,
  'g',
  29.0,
  'g',
  23.0,
  'g',
  7.0,
  'g',
  17.0,
  'g',
  0.0,
  'g'),
 ('3250393480096',
  202.0,
  'kcal',
  5.199999809265137,
  'g',
  2.700000047683716,
  'g',
  30.0,
  'g',
  1.899999976158142,
  'g',
  2.5,
  'g',
  7.5,
  'g',
  1.0,
  'g'),
 ('0000554004509',
  279.0,
  'kcal',
  6.300000190734863,
  'g',
  0.5,
  'g',
  48.20000076293945,
  'g',
  4.400000095367432,
  'g',
  5.800000190734863,
  'g',
  4.400000095367432,
  'g',
  1.090000033378601,
  'g'),
 ('3706228437938',
  399.39898681640625,
  'kcal',
  0.0,
  'g',
  0.0,
  'g',
  99.8998031616211,
  'g',
  99.8998031616211,
  'g',
  0.0,
  'g',
  0.0,
  'g',
  0.005405399948358536,
  'g'),
 ('45331053',
  297.0,
  'kcal',
  30.0,
  'g',
  19.0,
  'g',
  4.3000

In [73]:
query = f"INSERT INTO nutriments ({', '.join(sql_cols)}) VALUES %s"

In [76]:
execute_values(
    cursor,
    query,
    rows,
    page_size=10_000,
)
conn.commit()

In [75]:
conn.rollback()

In [ ]:
# Fermer la connexion
# cursor.close()
# conn.close()